In [ ]:
import json
import os
import pprint

import dotenv
import requests

In [ ]:
dotenv.load_dotenv()
OPENSEARCH_HOST = "https://localhost:10004"
OPENSEARCH_USER = os.getenv("OPENSEARCH_USER")
OPENSEARCH_PASSWORD = os.getenv("OPENSEARCH_PASSWORD")

### 検索する

##### agentic検索
##### https://docs.opensearch.org/latest/search-plugins/search-pipelines/agentic-query-translator-processor#usage

In [ ]:
OPENSEARCH_INDEX_NAME = "bra_panty_product_database"
OPENSEARCH_PIPELINE_NAME = "my-bra-and-panty-agentic-search-pipeline"

In [ ]:
agentic_search_payload = {"_source": ["product",
                                      "category",
                                      "name",
                                      "description",
                                      "detail",
                                      "image_url",
                                      "product_url",
                                      "size_price"],
                          "size": 5,
                          "query": {"agentic": {"query_text": "鮮やかな色でセクシーなショーツを提案頂けますか？"}}}

In [ ]:
agentic_search_url = "{a}/{b}/_search?search_pipeline={c}".format(a=OPENSEARCH_HOST,
                                                                  b=OPENSEARCH_INDEX_NAME,
                                                                  c=OPENSEARCH_PIPELINE_NAME)
response = requests.post(url=agentic_search_url,
                         auth=(OPENSEARCH_USER,
                               OPENSEARCH_PASSWORD),
                         headers={"Content-Type": "application/json"},
                         json=agentic_search_payload,
                         verify=False)
print(response.status_code)
pprint.pprint(response.json())

### FastAPI向け

##### FastAPI開発コード

In [ ]:
import pydantic

In [ ]:
class SearchRequest(pydantic.BaseModel):
    query: str = pydantic.Field(description="""
    検索したい商品の内容を自然文で入力します。
    例:
    - ワンポイントの装飾があるようなショーツを探してます。Tバックでお願いします。
    - 15,000円以内で、明るめな色で鮮やかなデザインのブラジャーを探してます。Bカップです。パッドの有るものをお願いします。谷間を作れるものがイイです。
    """)

In [ ]:
def search(req: SearchRequest):
    try:
        OPENSEARCH_INDEX_NAME = "bra_panty_product_database"
        OPENSEARCH_PIPELINE_NAME = "my-bra-and-panty-agentic-search-pipeline"
        agentic_search_payload = {"_source": ["product",
                                              "category",
                                              "name",
                                              "description",
                                              "detail",
                                              "image_url",
                                              "product_url",
                                              "size_price"],
                                  "size": 5,
                                  "query": {"agentic": {"query_text": req.query}}}
        agentic_search_url = "{a}/{b}/_search?search_pipeline={c}".format(a=OPENSEARCH_HOST,
                                                                          b=OPENSEARCH_INDEX_NAME,
                                                                          c=OPENSEARCH_PIPELINE_NAME)
        response = requests.post(url=agentic_search_url,
                                 auth=(OPENSEARCH_USER,
                                       OPENSEARCH_PASSWORD),
                                 headers={"Content-Type": "application/json"},
                                 json=agentic_search_payload,
                                 verify=False
                                )
        formatted_answer_json_dict = {}
        formatted_answer_json_dict["dsl_query"] = json.loads(response.json()["ext"]["dsl_query"])
        search_results_list = []
        num = 1
        for obj in response.json()["hits"]["hits"]:
            search_result_dict = {}
            search_result_dict["rank"] = num
            search_result_dict["score"] = obj["_score"]
            search_result_dict["product"] = obj["_source"]["product"]
            search_result_dict["category"] = obj["_source"]["category"]
            search_result_dict["name"] = obj["_source"]["name"]
            search_result_dict["description"] = obj["_source"]["description"]
            search_result_dict["detail"] = obj["_source"]["detail"]
            search_result_dict["image_url"] = "![{a}]({b})".format(a=obj["_source"]["product"], b=obj["_source"]["image_url"])
            search_result_dict["product_url"] = obj["_source"]["product_url"]
            search_result_dict["size_price"] = obj["_source"]["size_price"]
            search_results_list.append(search_result_dict)
            num = int(num) + 1
        formatted_answer_json_dict["results"] = search_results_list
        return json.dumps(formatted_answer_json_dict,
                          ensure_ascii=False)
    except Exception as e:
        return {"message": str(e)}

In [ ]:
search_request = SearchRequest(query="10,000円程度で、明るめな色でオシャレなものを探してます。")
search_response = search(req=search_request)

In [ ]:
pprint.pprint(json.loads(search_response))

##### FastAPIテスト

In [ ]:
fastapi_url = "http://localhost:10002/search_bra_and_panty_by_opensearch_agentic_search"

In [ ]:
user_search_query = "10,000円以下で、明るめな色でオシャレなショーツを探してます。"

In [ ]:
response = requests.post(url=fastapi_url,
                         headers={"Content-Type": "application/json"},
                         json={"query": user_search_query})
print(response.status_code)
pprint.pprint(response.json())